In [1]:
import torch
import pandas as pd
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

/Users/marconatale/Documents/GitHub/Magistrale/MNLP/HW2/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/marconatale/Documents/GitHub/Magistrale/MNLP/HW2/.venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
device = torch.device("mps") if torch.backends.mps.is_available() else \
        torch.device("cuda") if torch.cuda.is_available() else \
        torch.device("cpu")

In [5]:
print(f"Using device: {device}")

Using device: cpu


# Loading the PHI-4 Model

In [4]:
model_name = "microsoft/Phi-4-mini-instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)
pipe = pipeline("text-generation", model=model, tokenizer=tokenizer, device_map="auto")

Loading checkpoint shards: 100%|██████████| 2/2 [00:11<00:00,  5.67s/it]
Device set to use cpu


# Reading the Dataset

We'll load the CSV file containing Old Italian sentences that need to be translated to Modern Italian

In [5]:
# Load dataset
df = pd.read_csv("dataset.csv")

# Display basic information
print("DataFrame shape:", df.shape)
print("\nFirst 5 rows:")
df.head()


DataFrame shape: (97, 4)

First 5 rows:


,Author,Date,Region,Sentence
0,Brunetto Latini,1260-61,fior.,quella guerra ben fatta l' opera perché etc. E...
1,Bono Giamboni,1292,fior.,"crudele, e di tutte le colpe pigli vendetta, c..."
2,Valerio Massimo (red. V1,1336,fior.,Non d' altra forza d' animo fue ornato Ponzio ...
3,Lucano volg. (ed. Marinoni),1330/40,prat.,Se questo piace a tutti e se 'l tempo hae biso...
4,Brunetto Latini,1260-61,fior.,Officio di questa arte pare che sia dicere app...


# Translating the Sentences (Zero shot & Instruction-Style)

In [6]:
""" prompts = [
    "You are an expert in historical Italian linguistics. Translate this archaic Italian sentence to modern Italian:\n{sentence}\nONLY provide the translation in your response, nothing else.", # prompt with role
    "Translate this archaic Italian sentence to modern Italian:\n{sentence}\nONLY provide the translation in your response, nothing else.", # prompt without role
    "Consider the following historical context:\nAuthor: {author}\nTime period: {date}\nGeographical region: {region}\n\nHere is an archaic Italian sentence to translate to modern Italian:\n{sentence}\nONLY provide the translation in your response, nothing else.", # prompt with context
    "Sei un esperto di linguistica storica italiana. Traduci questa frase in italiano antico (1300) in italiano moderno:\n{sentence}\nRispondi con SOLO la traduzione", # prompt in italian
] """

prompts = [
    [
        {"role": "system", "content": "You are an expert in historical Italian linguistics. Only provide the translation in your response, nothing else."},
        {"role": "user", "content": "Translate this archaic Italian sentence to modern Italian:\n{sentence}"}
    ],
    [
        {"role": "system", "content": "Only provide the translation in your response, nothing else."},
        {"role": "user", "content": "Translate this archaic Italian sentence to modern Italian:\n{sentence}"}
    ],
    [
        {"role": "system", "content": "Only provide the translation in your response, nothing else."},
        {"role": "user", "content": "Consider the following historical context:\nAuthor: {author}\nTime period: {date}\nGeographical region: {region}\n\nHere is an archaic Italian sentence to translate to modern Italian:\n{sentence}"}
    ],
    [
        {"role": "system", "content": "Sei un esperto di linguistica storica italiana. Rispondi con SOLO la traduzione."},
        {"role": "user", "content": "Traduci questa frase in italiano antico (1300) in italiano moderno:\n{sentence}"}
    ]
]


In [9]:
def translate(df, prompt):
    translations = []
    for _, row in tqdm(df.iterrows(), total=df.shape[0]):
        # Create a copy of the prompt messages to avoid modifying the original
        formatted_prompt = []
        for message in prompt:
            message_copy = message.copy()
            # Replace placeholders in the content
            if "{sentence}" in message_copy["content"]:
                if any(placeholder in message_copy["content"] for placeholder in ["{author}", "{date}", "{region}"]):
                    message_copy["content"] = message_copy["content"].format(
                        sentence=row['Sentence'],
                        author=row['Author'],
                        date=row['Date'],
                        region=row['Region']
                    )
                else:
                    message_copy["content"] = message_copy["content"].format(sentence=row['Sentence'])
            formatted_prompt.append(message_copy)
        
        result = pipe(formatted_prompt, max_new_tokens=100, return_full_text=False)
        translation = result[0]['generated_text'].strip()
        translations.append(translation)
    return translations

In [ ]:
test_df = df.head(3)  # Using just the first 3 rows for testing

# Create a copy of the test dataframe to store results
test_results = test_df.copy()

print("Testing translation with each prompt...")
for idx, prompt in enumerate(prompts):
    test_col_name = f"Test_Modern_Italian_{idx}"
    test_results[test_col_name] = translate(test_df, prompt)
    
    # Display sample results
    print("\nSample results:")
    for i, row in test_results.iterrows():
        print(f"Original: {row['Sentence'][:50]}...")
        print(f"Translation: {row[test_col_name][:50]}...")
        print("---")

In [8]:
# For each prompt, translate all sentences and add as a new column in df
for idx, prompt in enumerate(prompts):
    col_name = f"Modern_Italian_{idx}"
    df[col_name] = translate(df, prompt)

100%|██████████| 97/97 [05:00<00:00,  3.10s/it]


## Saving the Results

We'll save the original sentences along with their translations to a new CSV file.

In [9]:
output_file = 'phi-4/phi-4_translations.csv'
df.to_csv(output_file, index=False)

print(f"Saved translations to {output_file}")

Saved translations to phi-4/phi-4_translations.csv


# Evaluation (Zero-Shot & Instruction-Style)

In [8]:
# Read the saved translations file
translations_df = pd.read_csv('phi-4/phi-4_translations.csv')

# Show the first few rows
print("\nFirst 5 rows:")
translations_df.head()


First 5 rows:


,Author,Date,Region,Sentence,Modern_Italian_0,Modern_Italian_1,Modern_Italian_2,Modern_Italian_3
0,Brunetto Latini,1260-61,fior.,quella guerra ben fatta l' opera perché etc. E...,Quella guerra ben fatta l' opera perché etc. E...,quella guerra ben fatta l' opera perché etc. E...,"quella guerra ben fatta l' opera perché, e dal...","Quella guerra ben fatta l' opera perché, e dal..."
1,Bono Giamboni,1292,fior.,"crudele, e di tutte le colpe pigli vendetta, c...","crudeli, e di tutte le colpe prende vendetta, ...","crudeli, e di tutte le colpe prende vendetta, ...","crudeli, e di tutte le colpe prende vendetta, ...","Crudele, e di tutte le colpe, prende vendetta,..."
2,Valerio Massimo (red. V1,1336,fior.,Non d' altra forza d' animo fue ornato Ponzio ...,Non d' altra forza d' animo fu ornato Ponzio A...,Non d' altra forza d' animo fu ornato Ponzio A...,Non d' altra forza d' animo fu ornato Ponzio A...,Non d' altra forza d' animo fu ornato Ponzio A...
3,Lucano volg. (ed. Marinoni),1330/40,prat.,Se questo piace a tutti e se 'l tempo hae biso...,Se questo ti piace a tutti e se il tempo ha bi...,Se questo ti piace a tutti e se il tempo ha bi...,Se questo ti piace a tutti e se il tempo ha bi...,Se questo piaccia a tutti e se 'l tempo abbia ...
4,Brunetto Latini,1260-61,fior.,Officio di questa arte pare che sia dicere app...,L'ufficio di questa arte sembra che sia detto ...,L'ufficio di questa arte sembra che sia detto ...,L'ufficio di questa arte sembra essere quello ...,L'ufficio di questa arte pare che sia detto ap...


In [3]:
import os
from dotenv import load_dotenv
from google import genai
import time

load_dotenv()
api_key = os.getenv("GEMINI_API_KEY")
client = genai.Client(api_key=api_key)

def evaluate_translation(row, translation_column, without_context = False) -> int:
    criteria = (
        "1. Completely unacceptable translation: the translation has no pertinence with the original meaning, the generated sentence is either gibberish or something that makes no sense.\n"
        "2. Severe semantic errors, omissions or substantial add ons on the original sentence. The errors are of semantic and syntactic nature. It’s still something no human would ever write.\n"
        "3. Partially wrong translation, the translation is lackluster, it contains errors, but are mostly minor errors, like typos, or small semantic errors.\n"
        "4. Good translation. The translation is mostly right, substantially faithful to the original text, but the style does not perfectly match the original sentence, still fluent and comprehensible, and could semantically acceptable.\n"
        "5. Perfect translation. The translation is accurate, fluent, complete and coherent. It retained the original meaning as much as it could."
    )

    if without_context:
        # Prompt without context
        prompt = (
            f"Rate the following translation on a scale of 1-5 based on these criteria:\n"
            f"{criteria}\n\n"
            f"Original sentence:\n\"{row['Sentence']}\"\n\n"
            f"Translated sentence:\n\"{row[translation_column]}\"\n\n"
            f"Provide only the rating (1-5)."
        )
    else:
        # Prompt with context
        prompt = (
            f"Rate this translation on a scale of 1-5 based on these criteria:\n"
            f"{criteria}\n\n"
            "Context:\n"
            f" • Author: {row['Author']}\n"
            f" • Date: {row['Date']}\n"
            f" • Region: {row['Region']}\n\n"
            f"Original sentence:\n\"{row['Sentence']}\"\n\n"
            f"Translated into Modern Italian:\n\"{row[translation_column]}\"\n\n"
            "Provide only the rating (1-5)."
        )
    resp = client.models.generate_content(
        model="gemini-2.0-flash",
        contents=prompt
    )
    try:
        return int(resp.text.strip())
    except ValueError:
        return None

In [4]:
def evaluate_translations_for_column(col: str, translations_df: pd.DataFrame, rate_limit_seconds: float = 4.5, filename = 'phi-4/phi-4_translations_with_eval.csv') -> pd.DataFrame:
    print(f"\nEvaluating translations for {col}...")
    
    # With context
    ratings = []
    for _, row in tqdm(translations_df.iterrows(), total=len(translations_df), desc=f"Evaluating {col} with context"):
        rating = evaluate_translation(row, col)
        ratings.append(rating)
        time.sleep(rate_limit_seconds)
    
    translations_df[f'{col}_gemini_eval'] = ratings
    
    # Without context
    ratings_no_context = []
    for _, row in tqdm(translations_df.iterrows(), total=len(translations_df), desc=f"Evaluating {col} without context"):
        rating = evaluate_translation(row, col, without_context=True)
        ratings_no_context.append(rating)
        time.sleep(rate_limit_seconds)
    
    translations_df[f'{col}_gemini_eval_no_context'] = ratings_no_context
    
    # Print summary statistics for this column
    print(f"\nEvaluation summary for {col}:")
    print(f"Average rating with context: {translations_df[f'{col}_gemini_eval'].mean():.2f}")
    print(f"Average rating without context: {translations_df[f'{col}_gemini_eval_no_context'].mean():.2f}")
    print(f"Difference: {(translations_df[f'{col}_gemini_eval'] - translations_df[f'{col}_gemini_eval_no_context']).mean():.2f}")
    
    # Save intermediate results
    translations_df.to_csv(filename, index=False)
    
    return translations_df

## Evaluating Translations from Prompt 0 (Role-based)

In [13]:
translations_df = evaluate_translations_for_column('Modern_Italian_0', translations_df)


Evaluating translations for Modern_Italian_0...


Evaluating Modern_Italian_0 without context: 100%|██████████| 97/97 [07:56<00:00,  4.91s/it]


Evaluation summary for Modern_Italian_0:
Average rating with context: 4.27
Average rating without context: 4.22
Difference: 0.05


## Evaluating Translations from Prompt 1 (No Role)

In [16]:
translations_df = evaluate_translations_for_column('Modern_Italian_1', translations_df)


Evaluating translations for Modern_Italian_1...


Evaluating Modern_Italian_1 without context: 100%|██████████| 97/97 [08:03<00:00,  4.99s/it]


Evaluation summary for Modern_Italian_1:
Average rating with context: 4.16
Average rating without context: 4.01
Difference: 0.15


## Evaluating Translations from Prompt 2 (With Context)

In [17]:
translations_df = evaluate_translations_for_column('Modern_Italian_2', translations_df)


Evaluating translations for Modern_Italian_2...


Evaluating Modern_Italian_2 without context: 100%|██████████| 97/97 [07:54<00:00,  4.89s/it]


Evaluation summary for Modern_Italian_2:
Average rating with context: 4.29
Average rating without context: 4.28
Difference: 0.01


## Evaluating Translations from Prompt 3 (Italian)

In [7]:
translations_df = evaluate_translations_for_column('Modern_Italian_3', translations_df)


Evaluating translations for Modern_Italian_3...


Evaluating Modern_Italian_3 without context: 100%|██████████| 97/97 [07:53<00:00,  4.89s/it]


Evaluation summary for Modern_Italian_3:
Average rating with context: 4.20
Average rating without context: 4.16
Difference: 0.03


# Translate the Sentences (Few-Shot)

In [11]:
# Load dataset
df = pd.read_csv("dataset.csv")
df.head()

,Author,Date,Region,Sentence
0,Brunetto Latini,1260-61,fior.,quella guerra ben fatta l' opera perché etc. E...
1,Bono Giamboni,1292,fior.,"crudele, e di tutte le colpe pigli vendetta, c..."
2,Valerio Massimo (red. V1,1336,fior.,Non d' altra forza d' animo fue ornato Ponzio ...
3,Lucano volg. (ed. Marinoni),1330/40,prat.,Se questo piace a tutti e se 'l tempo hae biso...
4,Brunetto Latini,1260-61,fior.,Officio di questa arte pare che sia dicere app...


In [13]:
few_shot_prompts = [
    [
        {"role": "system", "content": "You are an expert in historical Italian linguistics. Only provide the translation in your response, nothing else."},
        {"role": "user", "content": "Translate this archaic Italian sentence to modern Italian:\n… incontente mandà per li diti gotti, a li quae dolcementi parlando procurava cum doçe parole mitigar la lor aspreça"},
        {"role": "assistant", "content": "… subito mandò a chiamare quei suddetti Goti e, parlando con dolcezza, cercava con parole gentili di mitigare la loro durezza"},
        {"role": "user", "content": "Translate this archaic Italian sentence to modern Italian:\n… en leto agrevò de infirmitè ma de sana mente e de bone volontà e dretamentre parlando, no se voiando partir de questo mondo senza testamento"},
        {"role": "assistant", "content": "… si aggravò a letto per la malattia, ma con mente lucida e buona volontà e, parlando rettamente, non voleva lasciare questo mondo senza fare testamento"},
        {"role": "user", "content": "Translate this archaic Italian sentence to modern Italian:\n{sentence}"}
    ],
    [
        {"role": "system", "content": "Only provide the translation in your response, nothing else."},
        {"role": "user", "content": "Translate this archaic Italian sentence to modern Italian:\n… incontente mandà per li diti gotti, a li quae dolcementi parlando procurava cum doçe parole mitigar la lor aspreça"},
        {"role": "assistant", "content": "… subito mandò a chiamare quei suddetti Goti e, parlando con dolcezza, cercava con parole gentili di mitigare la loro durezza"},
        {"role": "user", "content": "Translate this archaic Italian sentence to modern Italian:\n… en leto agrevò de infirmitè ma de sana mente e de bone volontà e dretamentre parlando, no se voiando partir de questo mondo senza testamento"},
        {"role": "assistant", "content": "… si aggravò a letto per la malattia, ma con mente lucida e buona volontà e, parlando rettamente, non voleva lasciare questo mondo senza fare testamento"},
        {"role": "user", "content": "Translate this archaic Italian sentence to modern Italian:\n{sentence}"}
    ],
    [
        {"role": "system", "content": "Only provide the translation in your response, nothing else."},
        {"role": "user", "content": "Consider the following historical context:\nAuthor: anonimo\nTime period: XIV secolo\nGeographical region: Italia centro-settentrionale\n\nHere is an archaic Italian sentence to translate to modern Italian:\n… incontente mandà per li diti gotti, a li quae dolcementi parlando procurava cum doçe parole mitigar la lor aspreça"},
        {"role": "assistant", "content": "… subito mandò a chiamare quei suddetti Goti e, parlando con dolcezza, cercava con parole gentili di mitigare la loro durezza"},
        {"role": "user", "content": "Consider the following historical context:\nAuthor: anonimo\nTime period: XIV secolo\nGeographical region: Italia nord-orientale\n\nHere is an archaic Italian sentence to translate to modern Italian:\n… en leto agrevò de infirmitè ma de sana mente e de bone volontà e dretamentre parlando, no se voiando partir de questo mondo senza testamento"},
        {"role": "assistant", "content": "… si aggravò a letto per la malattia, ma con mente lucida e buona volontà e, parlando rettamente, non voleva lasciare questo mondo senza fare testamento"},
        {"role": "user", "content": "Consider the following historical context:\nAuthor: {author}\nTime period: {date}\nGeographical region: {region}\n\nHere is an archaic Italian sentence to translate to modern Italian:\n{sentence}"}
    ],
    [
        {"role": "system", "content": "Sei un esperto di linguistica storica italiana. Rispondi con SOLO la traduzione."},
        {"role": "user", "content": "Traduci questa frase in italiano antico (1300) in italiano moderno:\n… incontente mandà per li diti gotti, a li quae dolcementi parlando procurava cum doçe parole mitigar la lor aspreça"},
        {"role": "assistant", "content": "… subito mandò a chiamare quei suddetti Goti e, parlando con dolcezza, cercava con parole gentili di mitigare la loro durezza"},
        {"role": "user", "content": "Traduci questa frase in italiano antico (1300) in italiano moderno:\n… en leto agrevò de infirmitè ma de sana mente e de bone volontà e dretamentre parlando, no se voiando partir de questo mondo senza testamento"},
        {"role": "assistant", "content": "… si aggravò a letto per la malattia, ma con mente lucida e buona volontà e, parlando rettamente, non voleva lasciare questo mondo senza fare testamento"},
        {"role": "user", "content": "Traduci questa frase in italiano antico (1300) in italiano moderno:\n{sentence}"}
    ]
]

In [14]:
for idx, prompt in enumerate(few_shot_prompts):
    col_name = f"Modern_Italian_{idx}"
    df[col_name] = translate(df, prompt)

100%|██████████| 97/97 [26:27<00:00, 16.37s/it]


In [7]:
output_file = 'phi-4/phi-4_translations_fewshot.csv'
df.to_csv(output_file, index=False)

print(f"Saved translations to {output_file}")

NameError: name 'df' is not defined

# Evaluation (Few-Shot)

In [8]:
translations_df = pd.read_csv('phi-4/phi-4_translations_fewshot.csv')

translations_df.head()

,Author,Date,Region,Sentence,Modern_Italian_0,Modern_Italian_1,Modern_Italian_2,Modern_Italian_3
0,Brunetto Latini,1260-61,fior.,quella guerra ben fatta l' opera perché etc. E...,"quella guerra fu ben combattuta, perché etc. A...",quella guerra l'operò bene perché etc. E dall'...,quella guerra fu ben combattuta perché ecc. E ...,"quella guerra fu magnifica perché, etc. E dall..."
1,Bono Giamboni,1292,fior.,"crudele, e di tutte le colpe pigli vendetta, c...","crudele, e li prende tutte le colpe, vendetta ...","crudele, e per tutte le colpe, prende vendetta...","crudeli, e di tutte le colpe, prende vendetta,...","crudele, e di tutte le colpe prende vendetta c..."
2,Valerio Massimo (red. V1,1336,fior.,Non d' altra forza d' animo fue ornato Ponzio ...,Non era ornato da qualunque forza d'animo Ponz...,Non altro fu la forza d'animo di Ponzio Aufidi...,"Non aveva altra forza d'animo, Ponzio Aufidian...",Non d' altra forza di anima fece Ponzio Aufidi...
3,Lucano volg. (ed. Marinoni),1330/40,prat.,Se questo piace a tutti e se 'l tempo hae biso...,Se tutti lo trovano piacevole e se il tempo ha...,Se questo piaccia a tutti e se 'l tempo ha bis...,Se a tutti piace questo e se il tempo ha bisog...,Se questo piacerebbe a tutti e se il tempo ave...
4,Brunetto Latini,1260-61,fior.,Officio di questa arte pare che sia dicere app...,L'obiettivo di questa arte è che sia dirmi in ...,L'obiettivo di questa arte è dire le cose esat...,Il compito di questa arte sembra essere quello...,Il compito di questa arte sembra che sia fare ...


## Evaluating Translations from Prompt 0 (Role-based)

In [20]:
translations_df = evaluate_translations_for_column('Modern_Italian_0', translations_df, filename='phi-4/phi-4_translations_fewshot_with_eval.csv')


Evaluating translations for Modern_Italian_0...


Evaluating Modern_Italian_0 without context: 100%|██████████| 97/97 [07:59<00:00,  4.94s/it]


Evaluation summary for Modern_Italian_0:
Average rating with context: 4.24
Average rating without context: 4.20
Difference: 0.04


## Evaluating Translations from Prompt 1 (No Role)

In [17]:
translations_df = evaluate_translations_for_column('Modern_Italian_1', translations_df, filename='phi-4/phi-4_translations_fewshot_with_eval.csv')


Evaluating translations for Modern_Italian_1...


Evaluating Modern_Italian_1 without context: 100%|██████████| 97/97 [07:54<00:00,  4.89s/it]


Evaluation summary for Modern_Italian_1:
Average rating with context: 4.16
Average rating without context: 4.02
Difference: 0.14


## Evaluating Translations from Prompt 2 (With Context)

In [18]:
translations_df = evaluate_translations_for_column('Modern_Italian_2', translations_df, filename='phi-4/phi-4_translations_fewshot_with_eval.csv')


Evaluating translations for Modern_Italian_2...


Evaluating Modern_Italian_2 without context: 100%|██████████| 97/97 [08:01<00:00,  4.97s/it]


Evaluation summary for Modern_Italian_2:
Average rating with context: 4.30
Average rating without context: 4.20
Difference: 0.10


## Evaluating Translations from Prompt 3 (Italian)

In [19]:
translations_df = evaluate_translations_for_column('Modern_Italian_3', translations_df, filename='phi-4/phi-4_translations_fewshot_with_eval.csv')


Evaluating translations for Modern_Italian_3...


Evaluating Modern_Italian_3 without context: 100%|██████████| 97/97 [08:00<00:00,  4.95s/it]


Evaluation summary for Modern_Italian_3:
Average rating with context: 4.22
Average rating without context: 4.11
Difference: 0.10
